In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
</style>
"""))

# ※ Quiz : 경주여행과 전주여행에 대해 최빈단어시각화와 유사도분석
- (1) naver open API를 활용하여 블로그에 "경주여행", "전주여행"을 각각 500건씩 검색하여 백업(data/quiz/naver.csv)
    * 파일 내용 : query, no, title, link, description, total_text(title + ' ' + description)
- (2) naver.csv에서 total_text를 품사태깅(naver_pos.csv)
    * 파일 내용 : query, no, token, pos
- (3) 명사만 추출(naver_pos_nouns.csv)
    * query, token, pos
- (4) 빈도분석 백업(naver_pos_nouns_count.csv)
    * token, 경주빈도, 전주빈도, 빈도합
- (5) 빈도 시각화(워드클라우드, Text.plot)
    * 이미지 저장
- (6) 단어간 거리 분석(Word2Vec)

## 1. 네이버 open API 활용하여 검색 추출
- query, no, title, link, description, total_text(title + ' ' + description)

In [2]:
# .env가져오기
from dotenv import load_dotenv
import os
load_dotenv()

True

In [9]:
# 네이버 개발자 센터에 있는 소스를 가져오기
# 네이버 검색 API 예제 - 블로그 검색
import os
import sys
import urllib.request
client_id = os.getenv('CLIENT_ID')
client_secret = os.getenv('CLIENT_SECRET')
encText = urllib.parse.quote("경주 여행")
url = "https://openapi.naver.com/v1/search/blog.json?query=" + encText # JSON 결과
# url = "https://openapi.naver.com/v1/search/blog.xml?query=" + encText # XML 결과
request = urllib.request.Request(url)
request.add_header("X-Naver-Client-Id",client_id)
request.add_header("X-Naver-Client-Secret",client_secret)
response = urllib.request.urlopen(request)
rescode = response.getcode()
if(rescode==200):
    response_body = response.read()
    print(response_body.decode('utf-8')[:200])
    # items = json.loads(response_body.decode('utf-8'))
else:
    print("Error Code:" + rescode)

{
	"lastBuildDate":"Thu, 03 Sep 2026 16:57:14 +0900",
	"total":2735034,
	"start":1,
	"display":10,
	"items":[
		{
			"title":"3월의 <b>경주여행<\/b>.",
			"link":"https:\/\/lje77777.tistory.com\/7132",
			"


In [10]:
# 문자->dict
import json
from html import unescape # description에 있는 &lt;(특수문자)를 <로 변경
import requests
import pandas as pd
import re  # 특수문자 제거 @@ _._

In [21]:
query = "경주 여행"
start = 1
#url = f"https://openapi.naver.com/v1/search/blog.json?query={query}&display=100&start={start}"
url = "https://openapi.naver.com/v1/search/blog.json"
params = {
    'query':query,
    'display':100,
    'start':start
}
headers = {
    "X-Naver-Client-Id":client_id,
    "X-Naver-Client-Secret":client_secret
}
response = requests.get(url, headers=headers, params=params)
# 문자 -> dict
# items = json.loads(response.text)['items']
items = response.json()['items']
items[4]

{'title': '[<b>여행</b>]울산&amp;<b>경주 여행</b> Part4 : 동궁과 월지, 빛누리정원, 성호리조트',
 'link': 'https://jinstistory.tistory.com/138',
 'description': '장생포는 고래문화특구 라는 이름으로 고래를 주제로 한 다채로운 관광지를 형성해두었는데 사실 고래박물관이나 생태체 jinstistory.tistory.com 2023.08.16 - [<b>여행</b>/국내<b>여행</b>] - [<b>여행</b>]울산&amp;<b>경주 여행</b> Part3 : <b>경주</b>... ',
 'bloggername': "Two Park's",
 'bloggerlink': 'https://jinstistory.tistory.com/',
 'postdate': '20230817'}

In [32]:
# title과 description의 <b>없애기, html의 특수문자없애기, 일반특수 없애기
item = items[4]
title = item['title'].replace('<b>', ' ').replace('</b>', ' ')
title = unescape(title)
title = re.sub(r'[^a-zA-Z0-9가-힣]', ' ', title)
description = item['description'].replace('<b>', ' ').replace('</b>', ' ')
description = unescape(description)
description = re.sub(r'[^a-zA-Z0-9가-힣]', ' ', description)
link = item['link']
totaltext = title + ' ' + description
print(totaltext)
print(link)

  여행  울산  경주 여행  Part4   동궁과 월지  빛누리정원  성호리조트 장생포는 고래문화특구 라는 이름으로 고래를 주제로 한 다채로운 관광지를 형성해두었는데 사실 고래박물관이나 생태체 jinstistory tistory com 2023 08 16     여행  국내 여행       여행  울산  경주 여행  Part3    경주     
https://jinstistory.tistory.com/138


In [28]:
# re 정규표현식을 이용해서 특수문자 없애기
title = '[여행] ## & ktx 타고 짱 ㅋㅋ ㅠㅠ'
re.sub(r'[^a-zA-Z0-9가-힣]', ' ', title)

' 여행       ktx 타고 짱      '

In [33]:
# 네이버 API 정보 및 검색 정보
from dotenv import load_dotenv
import os
load_dotenv()
client_id = os.getenv('CLIENT_ID')
client_secret = os.getenv('CLIENT_SECRET')

queries = ['경주 여행', '전주 여행']
max_start = 5

In [ ]:
def get_search_item_return(query, start):
    'query와 start로 naver 블로그 검색한 결과로 title, link, descript, total_text를 dict list'
    
    pass

In [ ]:
for query in queries:
    for start in range(1, max_start+1):
        get_search_item_return(query, start)